In [1]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("gold-layer").getOrCreate()

SILVER_PATH = "/home/jovyan/work/submission/output/silver"
GOLD_PATH = "/home/jovyan/work/submission/output/gold"

In [4]:
orders_df = spark.read.parquet(f"{SILVER_PATH}/orders")
order_items_df = spark.read.parquet(f"{SILVER_PATH}/order_items")
customers_df = spark.read.parquet(f"{SILVER_PATH}/customers")
products_df = spark.read.parquet(f"{SILVER_PATH}/products")
sellers_df = spark.read.parquet(f"{SILVER_PATH}/sellers")
payments_df = spark.read.parquet(f"{SILVER_PATH}/payments")

In [11]:
from pyspark.sql.functions import sha2, concat_ws

dim_customers = customers_df.withColumn(
    "customer_key",
    sha2(concat_ws("||", col("customer_id")), 256)
)

dim_customers = dim_customers.select(
    "customer_key",
    "customer_id",
    "customer_city",
    "customer_state"
).dropDuplicates()

dim_customers.write.mode("overwrite").parquet(f"{GOLD_PATH}/dim_customers")

In [12]:
dim_products = products_df.withColumn(
    "product_key",
    sha2(concat_ws("||", col("product_id")), 256)
)

dim_products = dim_products.select(
    "product_key",
    "product_id",
    "product_category_name"
).dropDuplicates()

dim_products.write.mode("overwrite").parquet(f"{GOLD_PATH}/dim_products")

In [13]:
dim_sellers = sellers_df.withColumn(
    "seller_key",
    sha2(concat_ws("||", col("seller_id")), 256)
)

dim_sellers = dim_sellers.select(
    "seller_key",
    "seller_id",
    "seller_city",
    "seller_state"
).dropDuplicates()

dim_sellers.write.mode("overwrite").parquet(f"{GOLD_PATH}/dim_sellers")

In [10]:
from pyspark.sql.functions import col, sha2, concat_ws, datediff

# Join all required tables
fact_order_items = order_items_df \
    .join(orders_df, "order_id") \
    .join(products_df, "product_id") \
    .join(sellers_df, "seller_id") \
    .join(payments_df, "order_id")

# Add delivery metrics
fact_order_items = fact_order_items.withColumn(
    "days_to_deliver",
    datediff(
        col("order_delivered_customer_date"),
        col("order_purchase_timestamp")
    )
).withColumn(
    "days_delivery_vs_estimate",
    datediff(
        col("order_delivered_customer_date"),
        col("order_estimated_delivery_date")
    )
).withColumn(
    "is_late_delivery",
    col("days_delivery_vs_estimate") > 0
)

# Add surrogate keys (deterministic hashing)
fact_order_items = fact_order_items.withColumn(
    "customer_key",
    sha2(concat_ws("||", col("customer_id")), 256)
).withColumn(
    "product_key",
    sha2(concat_ws("||", col("product_id")), 256)
).withColumn(
    "seller_key",
    sha2(concat_ws("||", col("seller_id")), 256)
)

# Select final columns (aligned with requirement)
fact_order_items = fact_order_items.select(
    col("order_id"),
    col("order_item_id"),
    col("customer_key"),
    col("product_key"),
    col("seller_key"),
    col("order_purchase_timestamp").alias("order_date"),
    col("order_status"),
    col("price"),
    col("freight_value"),
    col("payment_value"),
    col("days_to_deliver"),
    col("days_delivery_vs_estimate"),
    col("is_late_delivery")
)

# Save
fact_order_items.write.mode("overwrite").parquet(f"{GOLD_PATH}/fact_order_items")

print("fact_order_items created")

fact_order_items created


In [14]:
print("fact:", fact_order_items.count())
print("customers:", dim_customers.count())
print("products:", dim_products.count())
print("sellers:", dim_sellers.count())

fact: 117601
customers: 99441
products: 32951
sellers: 3095
